# Resilient Plant · 01 · Metals Peer Benchmark

**Calibration anchors for the project model.** Load 5 years of US-listed financials, filter to the **Basic Materials sector** (276 metals/mining companies per year), compute clean sector-median ratios that become the defensible inputs for the capital-budgeting model in Notebook 03.

**Dataset:** Kaggle `200-financial-indicators-of-us-stocks-20142018`. We use 1,201 valid metals company-years (2014–2018) after filtering for revenue > $1M.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 03/
#     │   ├── dataset/                   (5 CSVs: 2014..2018_Financial_Data.csv)
#     │   └── resilient_plant/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here, or in a notebooks/ subfolder)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "resilient_plant":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 yearly CSVs — accept dataset/ or data/
YEARS = [2014, 2015, 2016, 2017, 2018]
csv_paths = {}
for y in YEARS:
    name = f"{y}_Financial_Data.csv"
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[y] = folder / name
            break
assert len(csv_paths) == 5, (
    f"Expected 5 CSVs (2014..2018_Financial_Data.csv) in {DATASET_DIR} or {DATA_DIR}; "
    f"found {sorted(csv_paths.keys())}"
)
print(f"Found 5 yearly CSVs : {sorted(csv_paths.keys())}")


## Step 1 — Load and pool 5 years of metals-sector data

In [ ]:
import pandas as pd
import numpy as np

frames = []
for year, path in csv_paths.items():
    df = pd.read_csv(path)
    df = df[df["Sector"] == "Basic Materials"].copy()
    df["year"] = year
    frames.append(df)

metals = pd.concat(frames, ignore_index=True)
print(f"Total Basic Materials company-years: {len(metals):,}")
print(f"Per-year breakdown:")
print(metals.groupby("year").size())


## Step 2 — Compute clean margins from raw P&L

The pre-computed `operatingProfitMargin` column is corrupted (every value = 1.0). We compute margins ourselves from `Operating Income / Revenue` etc.

In [ ]:
metals["gross_margin"]  = metals["Gross Profit"]     / metals["Revenue"]
metals["op_margin"]     = metals["Operating Income"] / metals["Revenue"]
metals["net_margin"]    = metals["Net Income"]       / metals["Revenue"]
metals["capex_to_rev"]  = -metals["Capital Expenditure"] / metals["Revenue"]
metals["tax_rate_eff"]  = metals["Income Tax Expense"] / metals["Earnings before Tax"]

# Replace infinities and filter to companies with meaningful revenue
for c in ["gross_margin", "op_margin", "net_margin", "capex_to_rev", "tax_rate_eff"]:
    metals[c] = metals[c].replace([np.inf, -np.inf], np.nan)

clean = metals[metals["Revenue"] > 1_000_000].copy()
print(f"Filtered to companies with revenue > $1M: {len(clean):,} company-years")


## Step 3 — Sector-median ratios by year

In [ ]:
ratios = ["gross_margin", "op_margin", "net_margin",
          "currentRatio", "quickRatio", "cashRatio",
          "debtRatio", "debtEquityRatio", "totalDebtToCapitalization",
          "returnOnAssets", "returnOnEquity",
          "capex_to_rev"]

# Filter outliers per ratio (winsorize at 1st/99th percentile)
clean_w = clean.copy()
for r in ratios:
    if r in clean_w.columns:
        lo, hi = clean_w[r].quantile([0.01, 0.99])
        clean_w[r] = clean_w[r].clip(lo, hi)

yearly_medians = clean_w.groupby("year")[ratios].median().round(3)
print("Sector-median ratios by year:")
print(yearly_medians.T.to_string())

yearly_medians.T.to_csv(OUTPUTS_DIR / "yearly_sector_medians.csv")


## Step 4 — Pooled 5-year calibration anchors

These are the model inputs for Notebook 03.

In [ ]:
# Tax rate uses only positive-EBT, positive-tax-paid observations (otherwise tax rate is meaningless)
tax_subset = clean[(clean["Earnings before Tax"] > 0) &
                   (clean["Income Tax Expense"] > 0)]

# CapEx/Revenue uses only the realistic 0-100% band (above 100% = data error)
capex_subset = clean[clean["capex_to_rev"].between(0, 1)]

# Debt/Equity uses only the realistic 0-5x band
de_subset = clean[clean["debtEquityRatio"].between(0, 5)]

anchors = {
    "Operating margin (median)"      : clean["op_margin"].median(),
    "Net margin (median)"            : clean["net_margin"].median(),
    "Gross margin (median)"          : clean["gross_margin"].median(),
    "CapEx / Revenue (median)"       : capex_subset["capex_to_rev"].median(),
    "Effective tax rate (median)"    : tax_subset["tax_rate_eff"].median(),
    "Debt / Equity (median)"         : de_subset["debtEquityRatio"].median(),
    "Current ratio (median)"         : clean["currentRatio"].median(),
    "Return on Assets (median)"      : clean["returnOnAssets"].median(),
    "Return on Equity (median)"      : clean["returnOnEquity"].median(),
}

print("=== POOLED 5-YEAR CALIBRATION ANCHORS ===")
print("(use these as default inputs in the capital-budgeting model)")
print()
for name, val in anchors.items():
    print(f"  {name:<35} {val:>7.3f}")

# Save anchors for downstream notebooks
anchors_df = pd.DataFrame(list(anchors.items()), columns=["anchor", "value"])
anchors_df.to_csv(DATA_DIR / "calibration_anchors.csv", index=False)
print(f"\nSaved anchors → {DATA_DIR / 'calibration_anchors.csv'}")

# Save the cleaned & winsorised dataset for reuse
clean_w.to_csv(DATA_DIR / "metals_cleaned.csv", index=False)
print(f"Saved cleaned metals dataset → {DATA_DIR / 'metals_cleaned.csv'}")


## Step 5 — Visualise sector-median trends

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

display_groups = [
    ("Profitability", ["gross_margin", "op_margin", "net_margin"], "Margin"),
    ("Liquidity",     ["currentRatio", "quickRatio", "cashRatio"], "Ratio"),
    ("Solvency",      ["debtRatio", "debtEquityRatio", "totalDebtToCapitalization"], "Ratio"),
    ("Returns",       ["returnOnAssets", "returnOnEquity"], "Return"),
    ("Capital intensity", ["capex_to_rev"], "Ratio"),
]

axes = axes.flatten()
palette = sns.color_palette("rocket_r", n_colors=4)

for i, (group_name, cols, ylabel) in enumerate(display_groups):
    if i >= len(axes):
        break
    ax = axes[i]
    for j, col in enumerate(cols):
        if col in yearly_medians.columns:
            ax.plot(yearly_medians.index, yearly_medians[col], marker="o", linewidth=2.5,
                    label=col, color=palette[j % len(palette)])
    ax.set_title(f"{group_name} · Sector Median 2014–2018", fontweight="bold", pad=10)
    ax.set_xlabel("Year"); ax.set_ylabel(ylabel)
    ax.legend(loc="best", frameon=True, fontsize=9)
    ax.set_xticks(yearly_medians.index)

# Hide last axis (we only have 5 groups)
axes[-1].axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "sector_median_trends.png", dpi=160, bbox_inches="tight")
plt.show()


## Step 6 — Cross-sectional dispersion (the 'envelope')

How spread-out is the sector each year? This is the volatility envelope the project must operate within.

In [ ]:
# Show 25th/50th/75th percentiles per year for op_margin and currentRatio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ratio, title in zip(axes, ["op_margin", "currentRatio"],
                             ["Operating Margin", "Current Ratio"]):
    pct = clean_w.groupby("year")[ratio].quantile([0.25, 0.5, 0.75]).unstack()
    ax.fill_between(pct.index, pct[0.25], pct[0.75],
                    alpha=0.3, color="#1F77B4", label="25th-75th percentile")
    ax.plot(pct.index, pct[0.5], marker="o", linewidth=2.5,
            color="#1F3864", label="Median")
    ax.set_title(f"{title} · Cross-sectional Dispersion", fontweight="bold", pad=10)
    ax.set_xlabel("Year"); ax.set_ylabel(title)
    ax.legend(loc="best")
    ax.set_xticks(pct.index)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dispersion_envelope.png", dpi=160, bbox_inches="tight")
plt.show()

print("\nThe envelope shows where the metals sector actually operates.")
print("The project's required performance must fit inside this envelope to be defensible.")


✅ **Notebook 01 complete.** Calibration anchors saved. Move to `02_ratio_analysis_envelope.ipynb`.